In [5]:
%pip install pandas
%pip install alpaca-py
%pip install pytz


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd

In [7]:
import pandas as pd
import numpy as np
#DATA gererated by AI TO CHECK THE CODE BELOW
np.random.seed(42)


tickers = {
    "AAA": 100,     
    "BBB": 50,      
    "CCC": 10,      
    "DDD": 200,     
    "EEE": 5        
}

start_date = pd.to_datetime("2025-01-02")
num_days = 3
minutes_per_day = 30  # simplified intraday

rows = []

for ticker, base_price in tickers.items():
    for d in range(num_days):
        date = start_date + pd.Timedelta(days=d)

        price = base_price

        for m in range(minutes_per_day):
            timestamp = date + pd.Timedelta(minutes=570 + m)  # 9:30am start

            # simulate small price movement
            change = np.random.normal(0, 0.2)
            open_price = price
            close_price = price + change
            high_price = max(open_price, close_price) + abs(np.random.normal(0, 0.1))
            low_price = min(open_price, close_price) - abs(np.random.normal(0, 0.1))

            # simulate different liquidity levels
            volume_multiplier = {
                "AAA": 5000,
                "BBB": 20000,
                "CCC": 800,
                "DDD": 40000,
                "EEE": 300
            }

            volume = int(abs(np.random.normal(volume_multiplier[ticker], volume_multiplier[ticker]*0.1)))

            rows.append([
                ticker,
                timestamp,
                round(open_price, 2),
                round(high_price, 2),
                round(low_price, 2),
                round(close_price, 2),
                volume
            ])

            price = close_price

columns = ["Ticker", "Date", "Open", "High", "Low", "Close", "Volume"]

df = pd.DataFrame(rows, columns=columns)

df.to_excel("test4stocks.xlsx", index=False)

print("Excel file 'test4stocks.xlsx' created successfully.")

Excel file 'test4stocks.xlsx' created successfully.


In [ ]:
import pandas as pd

# Load Excel
#df = pd.read_excel("test4stocks.xlsx") #can be changed to whatever type we want 
df = pd.read_parquet("1000Stocks.parquet")
df = df.dropna() # clean empty lines
df["date"] = pd.to_datetime(df["date"]) # converts data from the fprm of YYYY MM DD Time type to pandas  

final_set = [] # empty list for dollar bars

def process_ticker(ticker_df):
    local_final = []

for ticker, ticker_df in df.groupby("ticker"):  #split per tickers --- apple, nvda etc
    for date, day_df in ticker_df.groupby("date"):  #similarly split by day
        day_df = day_df.sort_values("date").reset_index(drop=True) # making sure the chronological order is fine and every piece starts as 0 index

    #Dollar Volume
        day_df["dollarVolume"] = day_df["close"] * day_df["volume"]

        # 50 bars per day target
        Strategist_Threshold = day_df["dollarVolume"].sum() / 50    #calculate the total dv per day and /50

        current_dollar_value = 0
        current_set = []

        #dollar bar
        for index, row in day_df.iterrows():

            dollar_tot = row["dollarVolume"]
            current_dollar_value += dollar_tot
            current_set.append(row)

            if current_dollar_value >= Strategist_Threshold:

                new_df_dollar_bar = pd.DataFrame(current_set) # new dataframe

                ticker_name = new_df_dollar_bar.iloc[0]["ticker"]
                open_price = new_df_dollar_bar.iloc[0]["close"]
                close_price = new_df_dollar_bar.iloc[-1]["close"]
                high_price = new_df_dollar_bar["close"].max()
                low_price = new_df_dollar_bar["close"].min()

                final_set.append({
                    "ticker": ticker_name,
                    "date": date,
                    "open": open_price,
                    "high": high_price,
                    "low": low_price,
                    "close": close_price,
                    "dollar volume": Strategist_Threshold
                })

           
                current_dollar_value -= Strategist_Threshold
                current_set = []
    break

# Final DataFrame
final_set_df = pd.DataFrame(final_set)
final_set_df.to_parquet("final_set.parquet")

print(final_set_df)
#2 core machine 

      ticker                      date      open      high       low  \
0       MSFT 2025-04-21 08:30:00-05:00  363.1500  363.1500  363.1500   
1       MSFT 2025-04-21 08:31:00-05:00  363.2950  363.2950  363.2950   
2       MSFT 2025-04-21 08:32:00-05:00  363.5100  363.5100  363.5100   
3       MSFT 2025-04-21 08:33:00-05:00  363.4900  363.4900  363.4900   
4       MSFT 2025-04-21 08:34:00-05:00  364.1900  364.1900  364.1900   
...      ...                       ...       ...       ...       ...   
79856   MSFT 2026-02-11 14:56:00-06:00  404.5996  404.5996  404.5996   
79857   MSFT 2026-02-11 14:57:00-06:00  404.7550  404.7550  404.7550   
79858   MSFT 2026-02-11 14:58:00-06:00  404.6000  404.6000  404.6000   
79859   MSFT 2026-02-11 14:59:00-06:00  404.2000  404.2000  404.2000   
79860   MSFT 2026-02-11 15:00:00-06:00  404.3700  404.3700  404.3700   

          close  dollar volume  
0      363.1500   2.079985e+06  
1      363.2950   5.133286e+05  
2      363.5100   5.671192e+05  
3  